# Spatio-Temporal ConvLSTM for Cellular Traffic Prediction: Telecom Italia Milano

Final-year project. A multi-layer ConvLSTM predicts future **internet traffic** on the 100×100 Milano
grid from the previous hour of traffic. It is compared with two naive baselines (persistence and
the historical window mean).

This notebook runs the whole pipeline on the **real** dataset:

1. find the project code and the raw daily files attached as Kaggle inputs
2. preprocess them into a `(T, C, 100, 100)` tensor
3. train the ConvLSTM on the GPU, with early stopping
4. evaluate it on the chronologically held-out test period and plot the results

Full documentation: `README.md` and `PROJECT_OVERVIEW.md` in the project folder.

In [ ]:
import glob, os, shutil, subprocess, sys, json
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
for d in ("/kaggle/working", "/kaggle/temp"):
    os.makedirs(d, exist_ok=True)
    print(f"{d}: {shutil.disk_usage(d).free / 1e9:.1f} GB free")
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator) before running."

## 1. Locate the project code and the raw data

In [ ]:
# Project code: the attached dataset that contains src/model.py (extract the zip if Kaggle did not)
hits = glob.glob("/kaggle/input/**/src/model.py", recursive=True)
if not hits:
    import zipfile
    for z in glob.glob("/kaggle/input/**/convlstm_milano*.zip", recursive=True):
        zipfile.ZipFile(z).extractall("/kaggle/temp/code")
    hits = glob.glob("/kaggle/temp/code/**/src/model.py", recursive=True)
assert hits, "Attach the project-code dataset (it must contain src/model.py)."
src_root = os.path.dirname(os.path.dirname(hits[0]))
PROJECT = "/kaggle/working/convlstm_milano"
shutil.copytree(src_root, PROJECT, dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("data", "outputs", "__pycache__", "*.pyc", "desktop.ini"))
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print("Project copied to", PROJECT, "->", sorted(os.listdir(PROJECT)))

# Raw daily files: sms-call-internet-mi-YYYY-MM-DD.(txt|csv)[.gz] anywhere under /kaggle/input
raw_files = sorted(f for f in glob.glob("/kaggle/input/**/sms-call-internet-mi-*", recursive=True)
                   if f.lower().endswith((".txt", ".csv", ".gz")))
assert raw_files, "Attach a dataset with the daily sms-call-internet-mi-* files."
raw_dirs = sorted({os.path.dirname(f) for f in raw_files})
assert len(raw_dirs) == 1, f"Raw files are spread over several folders: {raw_dirs}"
RAW_DIR = raw_dirs[0]
print(f"{len(raw_files)} raw files in {RAW_DIR}")
print("first:", os.path.basename(raw_files[0]), "| last:", os.path.basename(raw_files[-1]))

In [ ]:
# Detect the time resolution (the original release is 10-minute; some mirrors are hourly).
import numpy as np, pandas as pd
from src.dataset import read_raw_file
first = read_raw_file(raw_files[0])
gap = pd.Series(np.sort(first["time"].unique())).diff().dropna().min()
INTERVAL = int(gap / pd.Timedelta(minutes=1))
print(f"{len(first):,} (time, cell) records in the first file; time step = {INTERVAL} min; "
      f"{first['cell_id'].nunique():,} distinct cells")
display(first.head())
del first

# Shared overrides for every step, so preprocessing, training and evaluation agree.
SET = f"paths.raw_dir={RAW_DIR} data.interval_minutes={INTERVAL}"
os.environ["PYTHONWARNINGS"] = "ignore"

def run(cmd):
    # Run a command, stream its output into the notebook, and fail loudly on error.
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed with exit code {p.returncode}")

## 2. Preprocess

Rows are summed over country codes and timestamps converted to Milan local time. Each `CellID` is
mapped to `(row, col)` on the 100×100 grid. All five features are stored. Missing records mean no
activity, so they are zero-filled.

In [ ]:
run(f"python -m src.dataset --config configs/kaggle.yaml --set {SET}")

In [ ]:
meta = json.load(open("/kaggle/temp/processed/meta.json"))
print("tensor shape (T, C, H, W):", meta["shape"], "| features:", meta["features"])
print("days:", meta["days"][0], "->", meta["days"][-1], f"({len(meta['days'])} days)")

## 3. Train

Chronological 70/15/15 split, min-max scaling fitted on the training period only, `T_in = 6` frames
in and `T_out = 1` frame out. The model is 2 ConvLSTM layers (hidden 64, 3×3) plus a convolutional
prediction head. Training uses masked MSE, Adam (lr 1e-3), ReduceLROnPlateau, early stopping
(patience 10) and mixed precision. Training stops after `max_hours` so that evaluation always has
time to run.

In [ ]:
run(f"python -m src.train --config configs/kaggle.yaml --set {SET}")

## 4. Evaluate on the test period

In [ ]:
run(f"python -m src.evaluate --config configs/kaggle.yaml --set {SET}")

In [ ]:
OUT = "/kaggle/working/outputs/convlstm_internet"
metrics = json.load(open(f"{OUT}/metrics.json"))
table = pd.DataFrame({name: res["overall"] for name, res in metrics.items() if name != "notes"}).T
table.index.name = "model"
display(table.round(4))
display(pd.read_csv(f"{OUT}/metrics_per_horizon.csv").round(4))
print(json.dumps(metrics["notes"], indent=2))

In [ ]:
from IPython.display import Image, display
for name in ("loss_curve.png", "heatmaps.png", "cell_timeseries.png"):
    print(name)
    display(Image(filename=f"{OUT}/{name}"))

## Outputs

Everything in `/kaggle/working/outputs/convlstm_internet` is saved with the notebook version:
`best_model.pt`, `metrics.json`, `metrics_per_horizon.csv`, `history.csv`, the logs and the plots.